# Ablation — Evaluate `gini_coefficient` (keep or drop feature from impact on validation metrics)

In the submitted draft of this paper, `gini_coefficient` had a bug that it was identically zero; the revision corrects it (`sp._gini`). It is the only
feature whose definition changed after academic review, so it alone gets a keep-or-drop test on the validation set.

**Pre-specified rule** (tracker, commit 72b2128, recorded before this run): train LightGBM with and without
the feature on 10 stratified group splits; retain it only if removing it lowers **validation** F1 on at
least 8 of the 10 splits (one-sided sign test, p ≈ 0.055). The test set is not used for the decision.

The LightGBM configuration is the one selected by the last completed validation-only search
(commit 82148f1): `num_leaves=511, learning_rate=0.05, subsample=0.7, reg_lambda=1`. This notebook runs
before the final search, because the search depends on the feature set it decides.

In [1]:
import warnings
import pandas as pd
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
DATA_DIR = './data'
from scipy.stats import binomtest
PARAMS = dict(num_leaves=511, learning_rate=0.05, subsample=0.7, reg_lambda=1.0)
ALL63 = list(sp.FEATS_SUBMITTED)   # the submitted 63, in the submitted order
WITHOUT = [f for f in ALL63 if f != 'gini_coefficient']
df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)
rows = []
for s in sp.SPLIT_SEEDS:
    S = sp.make_splits(df, ALL63, method='group', seed=s)
    w = sp.lgbm_fit_eval(S, ALL63, PARAMS); wo = sp.lgbm_fit_eval(S, WITHOUT, PARAMS)
    rows.append(dict(split_seed=s, val_f1_with=w['val_f1'], val_f1_without=wo['val_f1'],
                     val_ap_with=w['val_ap'], val_ap_without=wo['val_ap']))
    print(s, round(w['val_f1'], 4), round(wo['val_f1'], 4), flush=True)
r = pd.DataFrame(rows)
r['f1_change_when_removed'] = r.val_f1_without - r.val_f1_with
r['ap_change_when_removed'] = r.val_ap_without - r.val_ap_with
print(r.round(4).to_string(index=False))

42 0.7017 0.7025
1 0.7164 0.7153
2 0.7147 0.7182
3 0.7091 0.7125
4 0.7067 0.704
5 0.7168 0.7221
6 0.7086 0.7097
7 0.7061 0.7041
8 0.7121 0.715
9 0.7099 0.7101
 split_seed  val_f1_with  val_f1_without  val_ap_with  val_ap_without  f1_change_when_removed  ap_change_when_removed
         42       0.7017          0.7025       0.7595          0.7599                  0.0009                  0.0003
          1       0.7164          0.7153       0.7696          0.7703                 -0.0011                  0.0008
          2       0.7147          0.7182       0.7742          0.7762                  0.0035                  0.0020
          3       0.7091          0.7125       0.7553          0.7594                  0.0033                  0.0042
          4       0.7067          0.7040       0.7493          0.7449                 -0.0027                 -0.0044
          5       0.7168          0.7221       0.7677          0.7688                  0.0053                  0.0011
          6    

In [2]:
k = int((r.f1_change_when_removed < 0).sum())
p = binomtest(k, len(r), 0.5, alternative='greater').pvalue
KEEP = k >= 8
print(f'Removing gini_coefficient lowered validation F1 on {k} of {len(r)} splits '
      f'(mean change {r.f1_change_when_removed.mean():+.4f}; one-sided sign test p = {p:.3f}).')
print('Decision:', 'RETAIN gini_coefficient' if KEEP else 'REMOVE gini_coefficient (62 features)')
sp.save_results([], '01_ablation_gini', extra=dict(
    rule='retain if removing lowers validation F1 on >= 8 of 10 group splits', splits_lowered=k,
    sign_test_p=p, mean_f1_change=float(r.f1_change_when_removed.mean()),
    mean_ap_change=float(r.ap_change_when_removed.mean()), decision='retain' if KEEP else 'remove',
    per_split=r.to_dict('records'), params=PARAMS))

Removing gini_coefficient lowered validation F1 on 3 of 10 splits (mean change +0.0011; one-sided sign test p = 0.945).
Decision: REMOVE gini_coefficient (62 features)
Saved results/01_ablation_gini.json
